# 01 · Collect & Clean

**Longevity Hook Lab, Part A**: *What makes a YouTube Short beat its channel's median views?*

This notebook turns the raw YouTube Data API pulls into one clean table, `data/shorts.csv`, with one row per Short and a binary label `beat`.

**How the raw data was collected** (`src/collect.py count`, run on 2026-10-23):
1. `channels.list(forHandle)` → channel ID and uploads playlist
2. `playlistItems.list` → every upload ID (1 quota unit per 50 videos)
3. `videos.list` → title, publish time, duration, views, likes, comments (1 unit per 50 videos)
4. For uploads ≤ 180 s, a request to `youtube.com/shorts/{id}`: **200 = Short**, redirect = regular video

Raw responses are cached as JSON in `data/raw/` (not committed to GitHub). Total quota used: ≈ 750 of the 10,000 free daily units.

In [1]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_colwidth", 80)

DATA = Path("../data")
RAW = DATA / "raw"
COLLECTION_DATE = pd.Timestamp("2026-10-23", tz="UTC")  # date of the API pull

MAX_SHORT_SECONDS = 180
MIN_AGE_DAYS = 14
CAP_PER_CHANNEL = 300
MIN_PER_CHANNEL = 75

## 1. Channels

11 channels, one per creator. The count check (2026-10-23) dropped @FoundMyFitness (7 Shorts) and @drmarkhyman (69 Shorts), so Rhonda Patrick and Mark Hyman are represented by their secondary channels.

In [2]:
channels = pd.read_csv(DATA / "channels.csv")
channels

,creator,handle,channel_type
0,Andrew Huberman,@hubermanlab,main
1,Peter Attia,@PeterAttiaMD,main
2,Rhonda Patrick,@FoundMyFitnessClips,secondary
3,Mark Hyman,@DrMarkHymanShorts,secondary
4,Eric Berg,@Drberg,main
5,David Perlmutter,@DavidPerlmutterMD,main
6,Jessie Inchauspe,@GlucoseRevolution,main
7,Louisa Nicola,@LouisaNicola,main
8,Gabrielle Lyon,@DrGabrielleLyon,main
9,Bryan Johnson,@BryanJohnson,main


## 2. Load the raw API data

One row per upload. `is_short` is the result of the `/shorts` check: `True`/`False`, or missing if the upload was longer than 180 s or wasn't among the newest 400 checked.

In [3]:
import re

def parse_duration(iso):
    """ISO 8601 duration ('PT1M5S') → seconds."""
    m = re.fullmatch(r"P(?:(\d+)D)?T?(?:(\d+)H)?(?:(\d+)M)?(?:(\d+)S)?", iso or "")
    if not m:
        return np.nan
    d, h, mi, s = (int(x) if x else 0 for x in m.groups())
    return d * 86400 + h * 3600 + mi * 60 + s


rows = []
for _, ch in channels.iterrows():
    stem = ch["handle"].lstrip("@").lower()
    videos = json.loads((RAW / f"{stem}_videos.json").read_text())["videos"]
    shorts_check = json.loads((RAW / f"{stem}_shorts_check.json").read_text())
    for v in videos:
        stats = v.get("statistics", {})
        rows.append({
            "video_id": v["id"],
            "creator": ch["creator"],
            "handle": ch["handle"],
            "channel_type": ch["channel_type"],
            "title": v["snippet"]["title"],
            "published_at": v["snippet"]["publishedAt"],
            "duration_sec": parse_duration(v["contentDetails"].get("duration")),
            "views": stats.get("viewCount"),
            "likes": stats.get("likeCount"),
            "comments": stats.get("commentCount"),
            "is_short": shorts_check.get(v["id"]),
        })

uploads = pd.DataFrame(rows)
uploads["published_at"] = pd.to_datetime(uploads["published_at"], utc=True)
for col in ["views", "likes", "comments"]:
    uploads[col] = pd.to_numeric(uploads[col])

print(uploads.shape)
uploads.head(3)

(16787, 11)


,video_id,creator,handle,channel_type,title,published_at,duration_sec,views,likes,comments,is_short
0,eYSOabR3YP0,Andrew Huberman,@hubermanlab,main,Essentials: Developing a Rational Approach to Supplementation for Health & P...,2026-10-22 12:00:33+00:00,2062,52777.0,592.0,62.0,None
1,2oqBvTRYcv0,Andrew Huberman,@hubermanlab,main,What Makes Great Leaders & How to Live an Extraordinary Life | Dr. Jared Dia...,2026-10-19 12:00:21+00:00,6800,141514.0,1332.0,95.0,None
2,YdIJW8RTptU,Andrew Huberman,@hubermanlab,main,Essentials: Neuralink & Technologies to Enhance Human Brains | Dr. Matt MacD...,2026-10-15 12:00:25+00:00,1818,128451.0,779.0,111.0,None


In [4]:
uploads.dtypes

video_id                        str
creator                         str
handle                          str
channel_type                    str
title                           str
published_at    datetime64[us, UTC]
duration_sec                  int64
views                       float64
likes                       float64
comments                    float64
is_short                     object
dtype: object

## 3. Filters

Each step is recorded in a funnel table so the drop-off is transparent.

| Step | Rule | Why |
|---|---|---|
| a | duration ≤ 180 s | YouTube Shorts can be up to 3 minutes |
| b | passed `/shorts` check | removes short *regular* videos and trailers |
| c | published ≥ 14 days before collection | younger Shorts are still gaining views |
| d | views not missing | some channels hide counts |
| e | English title | YouTube's language tags are unreliable here (English titles tagged Latvian/Hebrew), so we check the title text: ≥ 90% of letters must be plain ASCII |
| f | drop duplicate video IDs | safety check |
| g | newest 300 per channel | stops prolific channels dominating |
| h | channel keeps ≥ 75 Shorts | a stable median needs enough videos |

In [5]:
def is_english(title, threshold=0.9):
    letters = [c for c in title if c.isalpha()]
    if not letters:
        return True
    return sum(c.isascii() for c in letters) / len(letters) >= threshold


funnel = []
def log_step(name, df):
    funnel.append({"step": name, "rows": len(df), "channels": df["handle"].nunique()})
    return df

df = log_step("all uploads", uploads)
df = log_step("a. ≤ 180 s", df[df["duration_sec"] <= MAX_SHORT_SECONDS])
df = log_step("b. passed /shorts check (newest 400 checked)", df[df["is_short"] == True])
df["age_days"] = (COLLECTION_DATE - df["published_at"]).dt.days
df = log_step("c. ≥ 14 days old", df[df["age_days"] >= MIN_AGE_DAYS])
df = log_step("d. views present", df[df["views"].notna()])
df = log_step("e. English title", df[df["title"].apply(is_english)])
df = log_step("f. unique video IDs", df.drop_duplicates("video_id"))
df = df.sort_values("published_at", ascending=False)
df = log_step("g. newest 300 per channel", df.groupby("handle").head(CAP_PER_CHANNEL))
counts = df["handle"].value_counts()
df = log_step("h. channels with ≥ 75", df[df["handle"].isin(counts[counts >= MIN_PER_CHANNEL].index)])

pd.DataFrame(funnel)

,step,rows,channels
0,all uploads,16787,11
1,a. ≤ 180 s,8236,11
2,b. passed /shorts check (newest 400 checked),3514,11
3,c. ≥ 14 days old,3402,11
4,d. views present,3402,11
5,e. English title,3402,11
6,f. unique video IDs,3402,11
7,g. newest 300 per channel,2740,11
8,h. channels with ≥ 75,2740,11


In [6]:
# Shorts kept per channel
df.groupby(["creator", "handle"]).size().rename("shorts").sort_values(ascending=False).to_frame()

,,shorts
creator,handle,
Bryan Johnson,@BryanJohnson,300
Eric Berg,@Drberg,300
Gabrielle Lyon,@DrGabrielleLyon,300
Gary Brecka,@ultimatehumanpodcast,300
Jessie Inchauspe,@GlucoseRevolution,300
Louisa Nicola,@LouisaNicola,300
Peter Attia,@PeterAttiaMD,300
Rhonda Patrick,@FoundMyFitnessClips,300
David Perlmutter,@DavidPerlmutterMD,127


## 4. Label: did the Short beat its neighbours?

`beat = 1` if a Short's views are **above the median of the 31 Shorts its channel posted around the same time** (15 before, itself, 15 after).

Why not simply compare with the channel's overall median? A first version did, and the EDA showed two problems:
1. **Age:** Shorts 14–60 days old beat the overall median only 33% of the time, because they were still collecting views.
2. **Channel growth/decline:** e.g. Bryan Johnson's recent Shorts beat his overall median 87% of the time, his oldest only 17%.

The overall-median label was therefore partly measuring *when* a Short was posted. Comparing each Short with its time-neighbours on the same channel fixes both: neighbours are a similar age and come from the same phase of the channel. It still removes channel size (Huberman vs smaller channels).

Details: the window is centred, so the newest/oldest Shorts of each channel use a shorter one-sided window (minimum 16 Shorts). Because the window includes the Short itself, the beat-rate lands slightly under 50%.

In [7]:
WINDOW = 31

df = df.sort_values(["handle", "published_at"])
df["neighbour_median_views"] = (
    df.groupby("handle")["views"]
      .transform(lambda v: v.rolling(WINDOW, center=True, min_periods=WINDOW // 2 + 1).median())
)
df["beat"] = (df["views"] > df["neighbour_median_views"]).astype(int)
df["views_vs_neighbours"] = df["views"] / df["neighbour_median_views"]

df["beat"].value_counts(normalize=True).round(3)

beat
0    0.52
1    0.48
Name: proportion, dtype: float64

In [8]:
# Per-channel check: channel sizes differ hugely, beat-rate stays near 0.5
df.groupby("creator").agg(
    shorts=("video_id", "size"),
    median_views=("views", "median"),
    beat_rate=("beat", "mean"),
).sort_values("median_views", ascending=False).round(2)

,shorts,median_views,beat_rate
creator,,,
Eric Berg,300,537462.0,0.50
Bryan Johnson,300,273183.0,0.47
Andrew Huberman,89,115532.0,0.49
Jessie Inchauspe,300,56221.0,0.50
Peter Attia,300,23127.5,0.47
Rhonda Patrick,300,21272.5,0.47
Gabrielle Lyon,300,8757.5,0.48
Gary Brecka,300,6151.0,0.47
Mark Hyman,124,5580.5,0.48


In [9]:
# Did it fix the age problem? Compare beat-rate by age: overall-median label vs neighbour label
df["overall_median_beat"] = (df["views"] > df.groupby("handle")["views"].transform("median")).astype(int)
age_bucket = pd.cut(df["age_days"], bins=[13, 60, 180, 365, 730, 10_000],
                    labels=["14–60 d", "2–6 mo", "6–12 mo", "1–2 yr", "2+ yr"])
df.groupby(age_bucket, observed=True)[["overall_median_beat", "beat"]].mean().round(2)   .rename(columns={"overall_median_beat": "overall-median label", "beat": "neighbour label (used)"})

,overall-median label,neighbour label (used)
age_days,,
14–60 d,0.33,0.46
2–6 mo,0.49,0.49
6–12 mo,0.57,0.48
1–2 yr,0.50,0.48
2+ yr,0.54,0.49


## 5. Save

`likes` and `comments` are kept **for EDA only**: they are only known after posting, so using them as features would leak the answer.

In [10]:
columns = [
    "video_id", "creator", "handle", "channel_type", "title", "published_at", "age_days",
    "duration_sec", "views", "likes", "comments", "neighbour_median_views", "views_vs_neighbours", "beat",
]
shorts = df[columns].sort_values(["creator", "published_at"]).reset_index(drop=True)
shorts.to_csv(DATA / "shorts.csv", index=False)
print(shorts.shape)

(2740, 14)


## 6. Data dictionary — `data/shorts.csv`

| Column | Type | Description |
|---|---|---|
| `video_id` | str | YouTube video ID (`youtube.com/shorts/{id}`) |
| `creator` | str | Creator name |
| `handle` | str | YouTube channel handle |
| `channel_type` | str | `main` or `secondary` (reference only, not a model feature) |
| `title` | str | Short title at collection time |
| `published_at` | datetime (UTC) | Publish timestamp |
| `age_days` | int | Days between publishing and collection (2026-10-23) |
| `duration_sec` | int | Length in seconds (≤ 180) |
| `views` | int | View count at collection |
| `likes` | float | Like count at collection (**EDA only**: post-publish) |
| `comments` | float | Comment count at collection (**EDA only**: post-publish) |
| `neighbour_median_views` | float | Median views of the 31 Shorts the channel posted around the same time (centred window, includes itself) |
| `views_vs_neighbours` | float | `views / neighbour_median_views` |
| `beat` | int | **Target**: 1 if `views > neighbour_median_views`, else 0 |

**Assumptions & risks**
- Views are a single snapshot. The neighbour label compares Shorts of similar age, which limits (but can't fully remove) this.
- The `/shorts` check was run on each channel's newest 400 uploads ≤ 180 s, which covers the 300 cap.
- The English check is a simple character rule, not a language model.